1

In [4]:
from dotenv import load_dotenv
from pathlib import Path
import os

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import SystemMessage, HumanMessage
from prompts import SYSTEM_ENTITY_RELATIONSHIP_EXTRACTION_PROMPT_1, HUMAN_ENTITY_RELATIONSHIP_EXTRACTION_PROMPT_1
from langchain_groq import ChatGroq
from langchain_core.output_parsers import JsonOutputParser

root = Path().resolve().parent
file_path = root / "_docs" / "dummytext.txt"

load_dotenv()
# SERVER_AI_URL = os.getenv("SERVER_AI_URL")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")


loader = TextLoader(file_path=file_path)
docs = loader.load()
print(f"documento cargado.")


text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(documents=docs)
print(f"numero de chunks: {len(chunks)}")

i = 1
print(f"\n chunk {i+1} \n")
chunk = chunks[i].page_content
print(chunk)
print("\n\n")

humna_message = HUMAN_ENTITY_RELATIONSHIP_EXTRACTION_PROMPT_1.format(text=chunk)

messages = [
    SystemMessage(content=SYSTEM_ENTITY_RELATIONSHIP_EXTRACTION_PROMPT_1),
    HumanMessage(content=humna_message)
]
model = "openai/gpt-oss-120b"
llm = ChatGroq(model=model, temperature=0.3, api_key=GROQ_API_KEY)

ai_response = llm.invoke(messages)
print(ai_response.content)
print("\n\n\n")
parser = JsonOutputParser()
json = parser.parse(ai_response.content)
print(json.keys())
for entity in json["entities"]:
    print(entity)
    print("\n")
print("\n\n")
for relationship in json["relationships"]:
    print(relationship)
    print("\n")




documento cargado.
numero de chunks: 47

 chunk 2 

In the idyllic village of Santa Caterina, amidst the rolling hills and sun-kissed landscapes of Sicily, lies the genesis of the Caruso family, a lineage intertwined with the island's rich culinary tapestry. The Carusos were not mere inhabitants of the land; they were the keepers of a culinary heritage that spanned generations. Each family member contributed their unique flair, crafting a narrative of flavors that reflected their diverse experiences and deep-seated love for food.



{
    "entities": [
        {
            "entity_name": "Santa Caterina",
            "entity_type": "LUGAR",
            "entity_description": "Idílico pueblo mencionado como el lugar donde se origina la familia Caruso."
        },
        {
            "entity_name": "Sicily",
            "entity_type": "LUGAR",
            "entity_description": "Isla italiana que alberga el pueblo de Santa Caterina y la tradición culinaria."
        },
        {
       

In [ ]:
from neo4j import GraphDatabase

uri = "bolt://localhost:7687"
auth = ("neo4j", "langchain")

driver = GraphDatabase.driver(uri, auth=auth)

with driver.session() as session:
    result = session.run(
        'MATCH (tom:Person {name: "Tom Hanks"}) RETURN tom'
    )
    for record in result:
        print(f"`record`: \n {record} \n")
        print(f"`record` type: \n {type(record)} \n")
        print("---"*12)
        print(f"`record[0]`: \n {record[0]} \n")
        print(f"`record[0]` type: \n {type(record[0])} \n")
        print("---"*12)
        print(dir(record[0]))
        print()
        print(record[0]._properties)
        


`record`: 
 <Record tom=<Node element_id='4:7d3b78da-dd44-4def-895f-7522ccbcdbd5:77' labels=frozenset({'Person'}) properties={'born': 1956, 'name': 'Tom Hanks'}>> 

`record` type: 
 <class 'neo4j._data.Record'> 

------------------------------------
`record[0]`: 
 <Node element_id='4:7d3b78da-dd44-4def-895f-7522ccbcdbd5:77' labels=frozenset({'Person'}) properties={'born': 1956, 'name': 'Tom Hanks'}> 

`record[0]` type: 
 <class 'neo4j.graph.Node'> 

------------------------------------
['__abstractmethods__', '__annotations__', '__class__', '__class_getitem__', '__contains__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__orig_bases__', '__parameters__', '__reduce__', '__reduce_ex__', '__repr__', '__reversed__', '__setattr__', '__sizeof__', '__slots__', '__str__', '

In [11]:
def create_entity(name, label, description):
    query = """
MERGE (e:{label} {{name: '{name}'}})
SET e.description = '{description}'
    """
    return query.format(name=name, label=label, description=description)


print(create_entity(name="thomas", label="trabajador", description="buen trabajador"))


MERGE (e:trabajador {name: 'thomas'})
SET e.description = 'buen trabajador'
    


In [14]:
def create_relationship(source, target, rel_type, description):
    query = """
MATCH (a {{name: '{source}'}})
MATCH (b {{name: '{target}'}})
MERGE (a)-[r:{rel_type}]->(b)
SET r.description = '{description}'
    """
    return query.format(source=source, target=target, rel_type=rel_type, description=description)

print(create_relationship(source='thomas', target='miller', rel_type='COMPAÑERO_DE', description='thomas y miller son compañeros de trabajo'))
    


MATCH (a {name: 'thomas'})
MATCH (b {name: 'miller'})
MERGE (a)-[r:COMPAÑERO_DE]->(b)
SET r.description = 'thomas y miller son compañeros de trabajo'
    


In [15]:
import json
from neo4j import GraphDatabase


# -------------------------
# Configuración conexión
# -------------------------
URI = "bolt://localhost:7687"
USER = "neo4j"
PASSWORD = "langchain"
AUTH = (USER, PASSWORD)

driver = GraphDatabase.driver(URI, auth=AUTH)

with open("grafo.json", "r", encoding="utf-8") as f:
    data = json.load(f)


# -------------------------
# Inserción en Neo4j
# -------------------------

with driver.session() as seccion:
    for entity in data["entities"]:
        seccion.run(
            create_entity(
                name=entity["entity_name"],
                label=entity["entity_type"],
                description=entity["entity_description"]
            )
        )
    
    for rel in data["relationships"]:
        seccion.run(
            create_relationship(
                source=rel["source_entity"],
                target=rel["target_entity"],
                rel_type=rel["relationship_type"],
                description=rel["relationship_description"]
            )
        )
driver.close()


2